# Napa vintage prediction: 2000–2016, with a 2000 vs 2016 comparison

Adapted from `project3.ipynb`. The **main model learns all available vintage years from 2000 through 2016**. The final analysis compares its predictions for actual 2000 (worst) and 2016 (best) wines. A separate optional binary classifier directly distinguishes 2000 from 2016.

Uses the original model families: metadata-only logistic regression and TF-IDF descriptions + metadata logistic regression. Wine quality scores (`points`) are not the target.

In [1]:
import pandas as pd

wine_df = pd.read_csv("winemag-data-130k-v2.csv", index_col=0)
wine_df["title"].head(10)

0                    Nicosia 2013 Vulkà Bianco  (Etna)
1        Quinta dos Avidagos 2011 Avidagos Red (Douro)
2        Rainstorm 2013 Pinot Gris (Willamette Valley)
3    St. Julian 2013 Reserve Late Harvest Riesling ...
4    Sweet Cheeks 2012 Vintner's Reserve Wild Child...
5    Tandem 2011 Ars In Vitro Tempranillo-Merlot (N...
6     Terre di Giurfo 2013 Belsito Frappato (Vittoria)
7                Trimbach 2012 Gewurztraminer (Alsace)
8    Heinz Eifel 2013 Shine Gewürztraminer (Rheinhe...
9    Jean-Baptiste Adam 2012 Les Natures Pinot Gris...
Name: title, dtype: object

In [2]:
wine_df["wine vintage"] = (
    wine_df["title"]
    .str.extract(r"\b((?:19|20)\d{2})\b", expand=False)
    .astype("Int64")
)

wine_df[["title", "wine vintage"]].head(10)

,title,wine vintage
0,Nicosia 2013 Vulkà Bianco (Etna),2013
1,Quinta dos Avidagos 2011 Avidagos Red (Douro),2011
2,Rainstorm 2013 Pinot Gris (Willamette Valley),2013
3,St. Julian 2013 Reserve Late Harvest Riesling ...,2013
4,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,2012
5,Tandem 2011 Ars In Vitro Tempranillo-Merlot (N...,2011
6,Terre di Giurfo 2013 Belsito Frappato (Vittoria),2013
7,Trimbach 2012 Gewurztraminer (Alsace),2012
8,Heinz Eifel 2013 Shine Gewürztraminer (Rheinhe...,2013
9,Jean-Baptiste Adam 2012 Les Natures Pinot Gris...,2012


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

selected_vintages = list(range(2000, 2017))
napa = (wine_df.loc[
    wine_df["region_2"].eq("Napa") &
    wine_df["wine vintage"].isin(selected_vintages),
    ["title", "description", "wine vintage", "variety", "region_1"]
].dropna(subset=["title", "wine vintage"])
 .drop_duplicates(subset="title").copy())

napa["description"] = napa["description"].fillna("").astype(str)
for col in ["variety", "region_1"]:
    napa[col] = napa[col].fillna("Unknown").astype(str)
napa["wine vintage"] = napa["wine vintage"].astype(int)

# Group on exactly the same normalization used in the leakage check.
groups = napa["description"].str.strip().str.lower()
# Missing/blank descriptions are not useful for text learning.
napa = napa.loc[groups.ne("")].copy()
groups = groups.loc[napa.index]

X = napa[["variety", "region_1", "description"]].copy()
y = napa["wine vintage"].copy()
print("Counts per year:")
print(y.value_counts().sort_index().to_string())
print("Total wines:", len(y))

# Find a grouped split that includes both focal years in train and test.
splitter = GroupShuffleSplit(n_splits=100, test_size=0.2, random_state=42)
split_found = False
for train_pos, test_pos in splitter.split(X, y, groups):
    train_years = set(y.iloc[train_pos])
    test_years = set(y.iloc[test_pos])
    if {2000, 2016}.issubset(train_years) and {2000, 2016}.issubset(test_years):
        split_found = True
        break
if not split_found:
    raise ValueError("Could not find a split containing both 2000 and 2016 in training and testing. Inspect year counts.")

X_train, X_test = X.iloc[train_pos].copy(), X.iloc[test_pos].copy()
y_train, y_test = y.iloc[train_pos].copy(), y.iloc[test_pos].copy()
assert set(groups.iloc[train_pos]).isdisjoint(set(groups.iloc[test_pos]))
print("\nTrain/test counts for 2000 and 2016:")
print(pd.DataFrame({"Train": y_train.value_counts(), "Test": y_test.value_counts()}).reindex([2000, 2016]).fillna(0).astype(int))
print(f"Training: {len(X_train)} | Testing: {len(X_test)}")

## Train the original model families across 2000–2016

In [ ]:
def clean_vintage_description(text):
    return re.sub(r"\b(?:18|19|20|21)\d{2}\b", " ", str(text)).lower()

metadata_cols = ["variety", "region_1"]
baseline_model = Pipeline([
    ("categories", OneHotEncoder(handle_unknown="ignore")),
    ("classifier", LogisticRegression(max_iter=1500, random_state=42))
])
baseline_model.fit(X_train[metadata_cols], y_train)

combined_vintage_pipeline = Pipeline([
    ("features", ColumnTransformer([
        ("text", TfidfVectorizer(
            preprocessor=clean_vintage_description,
            lowercase=False, min_df=2, max_features=5000
        ), "description"),
        ("metadata", OneHotEncoder(handle_unknown="ignore"), metadata_cols)
    ])),
    ("classifier", LogisticRegression(max_iter=1500, random_state=42))
])

# Rare classes can make five-fold stratification impossible.
min_train_count = int(y_train.value_counts().min())
if min_train_count >= 2:
    folds = min(5, min_train_count)
    combined_search = GridSearchCV(
        combined_vintage_pipeline,
        {"classifier__C": [0.03, 0.1, 0.3, 1.0]},
        scoring="f1_macro",
        cv=StratifiedKFold(n_splits=folds, shuffle=True, random_state=42),
        n_jobs=-1, error_score="raise"
    )
    combined_search.fit(X_train, y_train)
    description_vintage_model = combined_search.best_estimator_
    print("Selected C:", combined_search.best_params_["classifier__C"])
else:
    # Keep all years, even if one has only a single training example.
    description_vintage_model = combined_vintage_pipeline
    description_vintage_model.fit(X_train, y_train)
    print("Skipped stratified CV because a vintage has only one training example.")

baseline_predictions = baseline_model.predict(X_test[metadata_cols])
description_vintage_predictions = description_vintage_model.predict(X_test)
print(pd.DataFrame([
    {"Model": "Metadata baseline",
     "Accuracy": accuracy_score(y_test, baseline_predictions),
     "Macro F1": f1_score(y_test, baseline_predictions, average="macro", zero_division=0)},
    {"Model": "Descriptions + metadata",
     "Accuracy": accuracy_score(y_test, description_vintage_predictions),
     "Macro F1": f1_score(y_test, description_vintage_predictions, average="macro", zero_division=0)}
]).round(3).to_string(index=False))

## Compare how the full 2000–2016 model predicts 2000 versus 2016

For each actual year, **exact-year recall** is the proportion correctly predicted as that year. Also compare absolute year error and the most common wrong predictions.

In [ ]:
results = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": description_vintage_predictions
}, index=y_test.index)
results["Absolute year error"] = (results["Actual"] - results["Predicted"]).abs()
rows = []
for year in [2000, 2016]:
    subset = results.loc[results["Actual"].eq(year)]
    predicted_as_year = results["Predicted"].eq(year)
    true_year = results["Actual"].eq(year)
    precision_denom = predicted_as_year.sum()
    rows.append({
        "Vintage": year,
        "Test wines": len(subset),
        "Correct": int((subset["Predicted"] == year).sum()),
        "Exact-year recall": (subset["Predicted"] == year).mean() if len(subset) else np.nan,
        "Precision": ((predicted_as_year & true_year).sum() / precision_denom
                      if precision_denom else 0.0),
        "Mean absolute year error": subset["Absolute year error"].mean()
    })
comparison_2000_2016 = pd.DataFrame(rows).set_index("Vintage")
print(comparison_2000_2016.round(3).to_string())
print("\nMost common predictions for actual 2000 and 2016:")
for year in [2000, 2016]:
    print(f"\nActual {year}:")
    print(results.loc[results.Actual.eq(year), "Predicted"].value_counts().head(10).to_string())

ax = comparison_2000_2016["Exact-year recall"].plot(
    kind="bar", figsize=(6, 4), rot=0, ylim=(0, 1),
    title="Correct vintage-year prediction: 2000 vs 2016"
)
ax.set_ylabel("Exact-year recall")
ax.set_xlabel("Actual vintage year")
plt.tight_layout()
plt.show()

## Optional: direct binary classification of 2000 versus 2016

This is a **different question** from the full 17-year model: among wines known to come from either 2000 or 2016, can the classifier tell which one? We use the same held-out split to avoid training/test leakage.

In [ ]:
binary_train = y_train.isin([2000, 2016])
binary_test = y_test.isin([2000, 2016])
X_train_binary, y_train_binary = X_train.loc[binary_train], y_train.loc[binary_train]
X_test_binary, y_test_binary = X_test.loc[binary_test], y_test.loc[binary_test]
if y_train_binary.nunique() < 2 or y_test_binary.nunique() < 2:
    print("Not enough examples of both years for a binary comparison.")
else:
    binary_model = Pipeline([
        ("features", ColumnTransformer([
            ("text", TfidfVectorizer(
                preprocessor=clean_vintage_description,
                lowercase=False, min_df=1, max_features=5000
            ), "description"),
            ("metadata", OneHotEncoder(handle_unknown="ignore"), metadata_cols)
        ])),
        ("classifier", LogisticRegression(max_iter=1500, random_state=42))
    ])
    binary_model.fit(X_train_binary, y_train_binary)
    binary_predictions = binary_model.predict(X_test_binary)
    print(classification_report(y_test_binary, binary_predictions,
                                labels=[2000, 2016], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_test_binary, binary_predictions, labels=[2000, 2016],
        cmap="Blues", colorbar=False
    )
    plt.title("Binary classifier: 2000 vs 2016")
    plt.tight_layout()
    plt.show()

### Interpretation caution

Different sample sizes, review wording, reviewer practices, and class imbalance can affect performance. A difference between 2000 and 2016 does **not** by itself prove vintage quality causes the difference. In the full model, report the test counts alongside exact-year recall.